In [ ]:
import os
import json
import pandas as pd
from collections import defaultdict, Counter

if os.path.exists('/workspace/data'):
    DATA_DIR, WORKSPACE_DIR = '/workspace/data', '/workspace'
elif os.path.exists('../environment/data'):
    DATA_DIR, WORKSPACE_DIR = '../environment/data', '..'
elif os.path.exists('environment/data'):
    DATA_DIR, WORKSPACE_DIR = 'environment/data', '.'
else:
    DATA_DIR, WORKSPACE_DIR = 'data', '.'

In [ ]:
tp_raw = pd.read_csv(f'{DATA_DIR}/touchpoints.csv', parse_dates=['timestamp'])
conv   = pd.read_csv(f'{DATA_DIR}/conversions.csv', parse_dates=['conversion_timestamp'])
cfg    = pd.read_csv(f'{DATA_DIR}/channel_config.csv')

lookback     = cfg.set_index('channel')['lookback_days'].to_dict()
cpc_map      = cfg.set_index('channel')['cost_per_click'].to_dict()
flat_fee_map = cfg.set_index('channel')['monthly_flat_fee'].to_dict()

print(f'touchpoints : {len(tp_raw):,}')
print(f'conversions : {len(conv):,}')
print(tp_raw['touchpoint_type'].value_counts().to_string())

In [ ]:
# Keep only clicks; impressions do not count for attribution
tp = (
    tp_raw[tp_raw['touchpoint_type'] == 'click']
    .copy()
    .sort_values(['user_id', 'timestamp'])
    .reset_index(drop=True)
)

In [ ]:
# Direct traffic suppression:
# A 'direct' touchpoint within 6 hours of the immediately preceding non-direct
# touchpoint for the same user is reassigned to that preceding channel.
rows = tp.to_dict('records')
last_non_direct = {}   # user_id -> {'channel': str, 'timestamp': Timestamp}

for row in rows:
    uid = row['user_id']
    ts  = row['timestamp']
    if row['channel'] == 'direct':
        prev = last_non_direct.get(uid)
        if prev is not None:
            diff_hours = (ts - prev['timestamp']).total_seconds() / 3600
            if diff_hours <= 6:
                row['channel'] = prev['channel']
    # Update anchor only after potential reassignment
    if row['channel'] != 'direct':
        last_non_direct[uid] = {'channel': row['channel'], 'timestamp': ts}

tp = pd.DataFrame(rows)
print('channel distribution after direct suppression:')
print(tp['channel'].value_counts().to_string())

In [ ]:
# Sort conversions chronologically per user and tag each with the prior conversion
# timestamp — this defines the isolation boundary for each path.
conv = conv.sort_values(['user_id', 'conversion_timestamp']).reset_index(drop=True)
conv['prev_conv_ts'] = conv.groupby('user_id')['conversion_timestamp'].shift(1)

In [ ]:
channel_revenue     = defaultdict(float)
attributed_channels = []

user_tp_groups = {uid: grp.reset_index(drop=True) for uid, grp in tp.groupby('user_id')}
lb_series      = pd.Series(lookback)

for _, c_row in conv.iterrows():
    uid     = c_row['user_id']
    conv_ts = c_row['conversion_timestamp']
    rev     = c_row['revenue']
    prev_ts = c_row['prev_conv_ts']

    user_tp = user_tp_groups.get(uid)
    if user_tp is None:
        continue

    # Path isolation
    mask = user_tp['timestamp'] < conv_ts
    if pd.notna(prev_ts):
        mask = mask & (user_tp['timestamp'] > prev_ts)
    path_tp = user_tp[mask]
    if path_tp.empty:
        continue

    # Per-channel lookback filter (vectorized)
    days_before = (conv_ts - path_tp['timestamp']).dt.days
    lb_vals     = path_tp['channel'].map(lb_series).fillna(30)
    eligible_df = path_tp[days_before <= lb_vals].copy()
    if eligible_df.empty:
        continue

    eligible_df = eligible_df.sort_values('timestamp')

    # Channel dedup: keep earliest per channel, re-sort by that timestamp
    deduped = (
        eligible_df
        .groupby('channel', sort=False)
        .first()
        .reset_index()
        .sort_values('timestamp')
        .reset_index(drop=True)
    )

    n = len(deduped)
    if   n == 1: weights = [1.0]
    elif n == 2: weights = [0.5, 0.5]
    else:
        mid_w   = 0.20 / (n - 2)
        weights = [0.40] + [mid_w] * (n - 2) + [0.40]

    for i, (_, td) in enumerate(deduped.iterrows()):
        channel_revenue[td['channel']] += rev * weights[i]
        attributed_channels.append(td['channel'])

print('attribution complete')
for ch, rv in sorted(channel_revenue.items()):
    print(f'  {ch}: {rv:,.2f}')

In [ ]:
# Spend model
# CPC channels : spend = cost_per_click × attributed click slots for that channel
# Flat-fee channels : spend = monthly_flat_fee × 3  (Q1 = Jan + Feb + Mar)
click_counts = Counter(attributed_channels)

all_channels = list(lookback.keys())
spend = {}
for ch in all_channels:
    cpc = cpc_map.get(ch, 0.0)
    fee = flat_fee_map.get(ch, 0.0)
    if cpc > 0:
        spend[ch] = round(cpc * click_counts.get(ch, 0), 2)
    elif fee > 0:
        spend[ch] = round(fee * 3, 2)
    else:
        spend[ch] = 0.0

roas = {}
for ch in all_channels:
    s = spend[ch]
    if s > 0:
        roas[ch] = round(channel_revenue.get(ch, 0.0) / s, 2)

print('spend:')
for ch, s in spend.items():
    print(f'  {ch}: {s:,.2f}  |  ROAS: {roas.get(ch, "n/a")}')

In [ ]:
report_rows = []
for ch in all_channels:
    s = spend[ch]
    report_rows.append({
        'channel':             ch,
        'attributed_revenue':  round(channel_revenue.get(ch, 0.0), 2),
        'spend':               s,
        'roas':                roas.get(ch, 0),
    })

report_df = pd.DataFrame(report_rows)
report_df.to_csv(f'{WORKSPACE_DIR}/attribution_report.csv', index=False)
report_df

In [ ]:
total_attributed_revenue          = float(round(sum(channel_revenue.values()), 2))
paid_search_attributed_revenue    = float(round(channel_revenue.get('paid_search', 0), 2))
display_attributed_revenue        = float(round(channel_revenue.get('display', 0), 2))
email_attributed_revenue          = float(round(channel_revenue.get('email', 0), 2))
organic_social_attributed_revenue = float(round(channel_revenue.get('organic_social', 0), 2))
direct_attributed_revenue         = float(round(channel_revenue.get('direct', 0), 2))
roas_paid_search                  = float(round(roas.get('paid_search', 0), 2))
roas_display                      = float(round(roas.get('display', 0), 2))
roas_email                        = float(round(roas.get('email', 0), 2))

print(f'total_attributed_revenue          = {total_attributed_revenue}')
print(f'paid_search_attributed_revenue    = {paid_search_attributed_revenue}')
print(f'display_attributed_revenue        = {display_attributed_revenue}')
print(f'email_attributed_revenue          = {email_attributed_revenue}')
print(f'organic_social_attributed_revenue = {organic_social_attributed_revenue}')
print(f'direct_attributed_revenue         = {direct_attributed_revenue}')
print(f'roas_paid_search                  = {roas_paid_search}')
print(f'roas_display                      = {roas_display}')
print(f'roas_email                        = {roas_email}')